# Week 50, Day 1: Graph representations

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w050d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
edges = [("Ada", "Bayo"), ("Bayo", "Chi"), ("Ada", "Chi"), ("Chi", "Dee")]

friends = {}
for a, b in edges:
    friends.setdefault(a, set()).add(b)
    friends.setdefault(b, set()).add(a)

print(sorted(friends["Chi"]))
print({name: len(f) for name, f in sorted(friends.items())})
print(sum(len(f) for f in friends.values()), len(edges))

In [ ]:
roads = [("Lagos", "Ibadan"), ("Ibadan", "Ilorin"), ("Lagos", "Abeokuta"), ("Abeokuta", "Ibadan")]
places = sorted({p for road in roads for p in road})
degree = {p: sum(p in road for road in roads) for p in places}
print(degree, sum(degree.values()) == 2 * len(roads))
print("possible undirected edges:", len(places) * (len(places) - 1) // 2, "actual:", len(roads))

In [ ]:
edges = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")]

# adjacency list: a dict of sets
adj = {}
for a, b in edges:
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set()).add(a)

# adjacency matrix: number the vertices first
names = sorted(adj)
index = {name: i for i, name in enumerate(names)}
n = len(names)
matrix = [[0] * n for _ in range(n)]          # n separate rows, not one row n times
for a, b in edges:
    matrix[index[a]][index[b]] = matrix[index[b]][index[a]] = 1

print({v: sorted(ns) for v, ns in sorted(adj.items())})
for name, row in zip(names, matrix):
    print(name, row)

In [ ]:
class Graph:
    def __init__(self, directed=False):
        self.directed = directed
        self._adj = {}

    def add_vertex(self, v):
        self._adj.setdefault(v, set())

    def add_edge(self, u, v):
        self.add_vertex(u)
        self.add_vertex(v)
        self._adj[u].add(v)
        if not self.directed:
            self._adj[v].add(u)

    def neighbours(self, v):
        return self._adj[v]

    def vertices(self):
        return self._adj.keys()

    def edge_count(self):
        total = sum(len(ns) for ns in self._adj.values())
        return total if self.directed else total // 2

follows = Graph(directed=True)
for a, b in [("ana", "ben"), ("ben", "ana"), ("ana", "cy"), ("dan", "cy")]:
    follows.add_edge(a, b)
print(sorted(follows.neighbours("ana")), follows.edge_count(), sorted(follows.vertices()))

In [ ]:
grid = ["..#.",
        ".##.",
        "...."]

def neighbours(cell):
    r, c = cell
    out = []
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != "#":
            out.append((nr, nc))
    return out

print(neighbours((0, 0)), neighbours((2, 1)), neighbours((0, 3)))

> **Key idea.** A graph is vertices joined by edges, which may be directed and may be weighted. Degree counts a vertex's edges (in- and out-degree in directed graphs), and the degrees of an undirected graph add up to twice the number of edges. Store graphs as edge lists (simple, how they arrive), adjacency matrices (O(V²) space, O(1) edge checks, good for small dense graphs) or adjacency lists (O(V + E), the default: a dict of sets, or of dicts for weights). Graph algorithms usually cost O(V + E) on adjacency lists. A graph class keeps the directed or undirected rule in one place. Implicit graphs, like grids and puzzle states, need only a neighbours function.

### Exercise: From edges to neighbours

Write `adjacency(edges, directed=False)`: a dict from every vertex to the **set** of its neighbours (vertices with no outgoing edges still appear, with an empty set). Then `degrees(adj)`: a dict from vertex to degree, and for directed graphs `in_degrees(adj)`: a dict from vertex to the number of edges arriving (0 for none).

In [ ]:
def adjacency(edges, directed=False):
    ___

def degrees(adj):
    ___

def in_degrees(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_build():\n    \"building\"\n    edges = [(\"A\", \"B\"), (\"A\", \"C\"), (\"B\", \"C\"), (\"C\", \"D\")]\n    adj = adjacency(edges)\n    assert adj == {\"A\": {\"B\", \"C\"}, \"B\": {\"A\", \"C\"}, \"C\": {\"A\", \"B\", \"D\"}, \"D\": {\"C\"}}\n    assert degrees(adj) == {\"A\": 2, \"B\": 2, \"C\": 3, \"D\": 1} and sum(degrees(adj).values()) == 2 * len(edges)\n    d = adjacency([(\"a\", \"b\"), (\"b\", \"a\"), (\"a\", \"c\"), (\"d\", \"c\")], directed=True)\n    assert d == {\"a\": {\"b\", \"c\"}, \"b\": {\"a\"}, \"c\": set(), \"d\": {\"c\"}}\n    assert degrees(d) == {\"a\": 2, \"b\": 1, \"c\": 0, \"d\": 1} and in_degrees(d) == {\"a\": 1, \"b\": 1, \"c\": 2, \"d\": 0}\n    assert adjacency([]) == {} and adjacency([(\"x\", \"y\"), (\"x\", \"y\")]) == {\"x\": {\"y\"}, \"y\": {\"x\"}}")

### Exercise: Matrix and list

Write `to_matrix(adj)`: a pair `(names, matrix)` where `names` is the sorted list of vertices and `matrix[i][j]` is 1 if there's an edge from `names[i]` to `names[j]`, else 0; and `from_matrix(names, matrix)`: the dict-of-sets adjacency list back. Each row must be its own list. Then `density(adj, directed)`: edges divided by the most possible (n(n − 1) directed, n(n − 1)/2 undirected; 0.0 when n < 2).

In [ ]:
def to_matrix(adj):
    ___

def from_matrix(names, matrix):
    ___

def density(adj, directed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_matrix():\n    \"conversions\"\n    adj = {\"A\": {\"B\", \"C\"}, \"B\": {\"A\", \"C\"}, \"C\": {\"A\", \"B\", \"D\"}, \"D\": {\"C\"}}\n    names, m = to_matrix(adj)\n    assert names == [\"A\", \"B\", \"C\", \"D\"] and m[2] == [1, 1, 0, 1] and m[3] == [0, 0, 1, 0]\n    m[0][1] = 9\n    assert m[1][1] == 0, \"each row its own list\"\n    assert from_matrix(*to_matrix(adj)) == adj\n    assert density(adj, False) == 4 / 6 and density({\"a\": set()}, True) == 0.0\n    rng = random.Random(1)\n    for _ in range(50):\n        vs = list(range(rng.randint(1, 7)))\n        g = {v: {w for w in vs if w != v and rng.random() < 0.4} for v in vs}\n        assert from_matrix(*to_matrix(g)) == g\n        assert density(g, True) == (sum(map(len, g.values())) / (len(vs) * (len(vs) - 1)) if len(vs) > 1 else 0.0)")

### Exercise: A graph class

Write `Graph(directed=False)` with `add_vertex(v)`, `add_edge(u, v, weight=1)` (adding missing vertices), `has_edge(u, v)`, `weight(u, v)` (`KeyError` if there's no such edge), `neighbours(v)` (a sorted list), `remove_edge(u, v)` (`KeyError` if absent), `remove_vertex(v)` (removing every edge touching it, in O(degree) for undirected graphs), `len(g)` (vertices) and `edge_count()`. Store a dict of dicts. Self-loops (`u == v`) raise `ValueError`.

In [ ]:
class Graph:
    def __init__(self, directed=False):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_graph():\n    \"a graph class\"\n    g = Graph()\n    g.add_edge(\"A\", \"B\", 5)\n    g.add_edge(\"B\", \"C\")\n    g.add_vertex(\"D\")\n    assert len(g) == 4 and g.edge_count() == 2 and g.has_edge(\"B\", \"A\") and g.weight(\"A\", \"B\") == 5 and g.neighbours(\"B\") == [\"A\", \"C\"]\n    g.remove_vertex(\"B\")\n    assert len(g) == 3 and g.edge_count() == 0 and g.neighbours(\"A\") == [] and not g.has_edge(\"A\", \"B\")\n    for f in (lambda: g.weight(\"A\", \"C\"), lambda: g.remove_edge(\"A\", \"C\")):\n        try:\n            f()\n            assert False\n        except KeyError:\n            pass\n    try:\n        g.add_edge(\"A\", \"A\")\n        assert False\n    except ValueError:\n        pass\n    d = Graph(directed=True)\n    d.add_edge(1, 2, 3)\n    d.add_edge(3, 1)\n    assert d.has_edge(1, 2) and not d.has_edge(2, 1) and d.edge_count() == 2\n    d.remove_vertex(1)\n    assert d.edge_count() == 0 and len(d) == 2\ndef test_random():\n    \"against a set of pairs\"\n    rng = random.Random(2)\n    for directed in (False, True):\n        g, ref = Graph(directed), set()\n        for _ in range(500):\n            u, v = rng.sample(range(8), 2)\n            if rng.random() < 0.6:\n                g.add_edge(u, v)\n                ref.add((u, v))\n                if not directed:\n                    ref.add((v, u))\n            elif g.has_edge(u, v):\n                g.remove_edge(u, v)\n                ref.discard((u, v))\n                if not directed:\n                    ref.discard((v, u))\n            assert g.edge_count() == (len(ref) if directed else len(ref) // 2)")

### Exercise: Reversing a directed graph

For a directed graph as a dict of sets, write `reverse(adj)`: the graph with every edge turned round (every vertex kept). Then `sources(adj)` (vertices with no incoming edges) and `sinks(adj)` (no outgoing edges), each a sorted list, and `mutual(adj)`: the sorted list of pairs `(u, v)` with `u < v` where the edge goes both ways.

In [ ]:
def reverse(adj):
    ___

def sources(adj):
    ___

def sinks(adj):
    ___

def mutual(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reverse():\n    \"reversing\"\n    follows = {\"ana\": {\"ben\", \"cy\"}, \"ben\": {\"ana\"}, \"cy\": set(), \"dan\": {\"cy\"}}\n    assert reverse(follows) == {\"ana\": {\"ben\"}, \"ben\": {\"ana\"}, \"cy\": {\"ana\", \"dan\"}, \"dan\": set()}\n    assert reverse(reverse(follows)) == follows\n    assert sources(follows) == [\"dan\"] and sinks(follows) == [\"cy\"] and mutual(follows) == [(\"ana\", \"ben\")]\n    assert sources({}) == [] and mutual({1: {2}, 2: {1, 3}, 3: {2}}) == [(1, 2), (2, 3)]")

### Exercise: A grid as a graph

For a grid given as a list of equal-length strings with `#` for walls, write `neighbours(grid, cell)` (open cells up, down, left and right, in that order) and `grid_edges(grid)`: the number of undirected edges between open cells, counted with the handshake lemma (half the sum of the neighbour counts). Then `dead_ends(grid)`: open cells with exactly one open neighbour, sorted.

In [ ]:
def neighbours(grid, cell):
    ___

def grid_edges(grid):
    ___

def dead_ends(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_grid():\n    \"grids\"\n    grid = [\"..#.\",\n            \".##.\",\n            \"....\"]\n    assert neighbours(grid, (0, 0)) == [(1, 0), (0, 1)] and neighbours(grid, (2, 1)) == [(2, 0), (2, 2)]\n    assert grid_edges(grid) == 8 and grid_edges([\"##\"]) == 0 and grid_edges([\"..\", \"..\"]) == 4\n    assert dead_ends(grid) == [(0, 1), (0, 3)]")

### Exercise: People you may know

In an undirected friendship graph (dict of sets), write `suggestions(adj, person, k)`: up to k people who aren't `person` and aren't already their friends, ranked by the number of **mutual friends** (most first, ties alphabetical), ignoring people with none in common. Count by walking friends of friends: O(sum of the friends' degrees), not by comparing with everyone. Then `common_count(adj, a, b)`: the number of mutual friends of a and b, using set intersection.

In [ ]:
def suggestions(adj, person, k):
    ___

def common_count(adj, a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_suggest():\n    \"suggestions\"\n    adj = {\"ana\": {\"ben\", \"cy\"}, \"ben\": {\"ana\", \"dan\", \"eve\"}, \"cy\": {\"ana\", \"dan\"}, \"dan\": {\"ben\", \"cy\", \"fay\"}, \"eve\": {\"ben\"}, \"fay\": {\"dan\"}}\n    assert suggestions(adj, \"ana\", 3) == [\"dan\", \"eve\"] and suggestions(adj, \"fay\", 5) == [\"ben\", \"cy\"]\n    assert common_count(adj, \"ana\", \"dan\") == 2 and common_count(adj, \"eve\", \"fay\") == 0\n    rng = random.Random(3)\n    for _ in range(50):\n        people = list(range(10))\n        g = {p: set() for p in people}\n        for _ in range(15):\n            a, b = rng.sample(people, 2)\n            g[a].add(b)\n            g[b].add(a)\n        p = rng.choice(people)\n        others = [q for q in people if q != p and q not in g[p] and len(g[p] & g[q]) > 0]\n        want = sorted(others, key=lambda q: (-len(g[p] & g[q]), q))[:4]\n        assert suggestions(g, p, 4) == want")

### Bug hunt: One-way friendships

This builds an undirected friendship graph but only records each edge in one direction, so Chi seems to have no friends: this prints `[] 2`. Record both directions so it prints `['Bayo'] 4`.

In [ ]:
edges = [("Ada", "Bayo"), ("Bayo", "Chi")]
adj = {}
for a, b in edges:
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set())
print(sorted(adj["Chi"]), sum(len(s) for s in adj.values()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both ways\"\n    expect_output(\"['Bayo'] 4\")")

### Bug hunt: Every row the same row

This builds an adjacency matrix with `[[0] * n] * n`, which makes n references to **one** row, so setting one cell sets the whole column: this prints `[1, 1, 1]`. Make each row its own list so it prints `[1, 0, 0]`.

In [ ]:
n = 3
matrix = [[0] * n] * n
matrix[0][1] = 1
print([row[1] for row in matrix])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate rows\"\n    expect_output(\"[1, 0, 0]\")")

### Bug hunt: Counting edges twice

This counts the edges of an undirected graph by adding up the neighbour sets, but each edge is in two sets, so the count is doubled: this prints `8`. Halve it (the handshake lemma) so it prints `4`.

In [ ]:
adj = {"A": {"B", "C"}, "B": {"A", "C"}, "C": {"A", "B", "D"}, "D": {"C"}}
print(sum(len(ns) for ns in adj.values()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"handshakes\"\n    expect_output(\"4\")")

### Puzzle: Building an adjacency list

Arrange the lines to build an undirected adjacency list from an edge list. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# An undirected graph with vertices 1 to 5 and edges 1-2, 1-3, 2-3, 3-4.

p = ___   # the degree of vertex 3
q = ___   # the sum of all the degrees
r = ___   # True or False: the graph is connected
s = ___   # the number of cells in its adjacency matrix
t = ___   # the most edges a simple undirected graph on 5 vertices can have

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 8 and r is False, \"p, q and r\"\n    assert s == 25 and t == 10, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reading a route map

A transport map arrives as text: one connection per line, either `A - B 12` (both ways, 12 minutes) or `A > B 5` (one way only), with station names that may contain spaces (`Old Market - River Gate 7`). Blank lines and lines starting with `#` are ignored. Write `load_routes(text)`, returning a weighted directed adjacency list (a dict of dicts; a two-way connection becomes two edges), and raising `ValueError` with the **line number** in the message for: a line with no ` - ` or ` > ` separator, a missing or non-positive whole-number time, an empty station name, a station connected to itself, or the same connection given twice (in either form, for the same direction). Then:

- `stations(adj)`: every station, sorted;
- `one_way_only(adj)`: sorted pairs `(a, b)` with an edge a → b but none b → a;
- `busiest(adj, k)`: the k stations with the most connections **out**, most first, ties alphabetical;
- `to_text(adj)`: text that `load_routes` turns back into the same graph, using `-` lines for pairs with the same time both ways (written once, the alphabetically smaller station first) and `>` lines otherwise, sorted.

In [ ]:
def load_routes(text):
    ___

def stations(adj):
    ___

def one_way_only(adj):
    ___

def busiest(adj, k):
    ___

def to_text(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("MAP = \"\"\"# the old town\nOld Market - River Gate 7\nRiver Gate - Station 4\n\nStation > Old Market 9\nOld Market > Station 11\nHill Top - Station 6\n\"\"\"\ndef test_load():\n    \"loading\"\n    adj = load_routes(MAP)\n    assert adj[\"Old Market\"] == {\"River Gate\": 7, \"Station\": 11} and adj[\"Station\"] == {\"River Gate\": 4, \"Old Market\": 9, \"Hill Top\": 6}\n    assert stations(adj) == [\"Hill Top\", \"Old Market\", \"River Gate\", \"Station\"]\n    assert one_way_only(adj) == [] and one_way_only(load_routes(\"A > B 2\")) == [(\"A\", \"B\")]\n    assert busiest(adj, 2) == [\"Station\", \"Old Market\"]\n    assert load_routes(to_text(adj)) == adj and load_routes(\"\") == {}\n    assert to_text(load_routes(\"B - A 3\\nA > C 1\")) == \"A - B 3\\nA > C 1\"\ndef test_errors():\n    \"bad lines\"\n    bad = [\"A = B 3\", \"A - B\", \"A - B x\", \"A - B 0\", \" - B 3\", \"A - A 3\", \"A - B 3\\nB > A 4\", \"A > B 1\\nA > B 2\"]\n    for text in bad:\n        try:\n            load_routes(text)\n            assert False, text\n        except ValueError as e:\n            assert \"line\" in str(e).lower(), str(e)\n    try:\n        load_routes(\"# fine\\nA - B 3\\nC - D\")\n        assert False\n    except ValueError as e:\n        assert \"3\" in str(e)")

## Warm-up

Four friendships, and the friend counts add up to 8: each friendship is counted once from each end. That's true of every undirected graph, and the first fact of graph theory: Leonhard Euler noticed it in 1736 while solving a puzzle about the bridges of Königsberg, the problem usually counted as the birth of the subject.

## Vertices and edges

A **graph** G = (V, E) is a set of **vertices** (or nodes) V and a set of **edges** E, each edge joining two vertices.

- **Undirected**: edges have no direction. Friendship: if Ada knows Bayo, Bayo knows Ada.
- **Directed**: each edge goes **from** one vertex **to** another. Following someone online, one-way streets, "task A must happen before task B".
- **Weighted**: each edge carries a number: a distance, a cost, a time.

```text
  undirected                 directed                  weighted (undirected)
  A --- B                    A ---> B                  A --5-- B
  |   / |                    ^      |                  |      /
  |  /  |                    |      v                  2     3
  C     D                    C <--- D                  |   /
                                                       C
```

The words you'll need:

- Two vertices joined by an edge are **adjacent**, or **neighbours**.
- The **degree** of a vertex is its number of edges; in a directed graph, **in-degree** counts edges arriving and **out-degree** edges leaving.
- A **path** is a sequence of vertices, each joined to the next by an edge; its **length** is the number of edges (or, in a weighted graph, often the total weight).
- A **cycle** is a path of at least one edge that returns to its start without repeating an edge (and, for undirected graphs, of at least three vertices).
- An undirected graph is **connected** if there's a path between every pair of vertices.
- A directed graph with no cycles is a **DAG** (directed acyclic graph). Friday's topological sort is about these.
- A **tree** is a connected undirected graph with no cycles; it always has exactly |V| − 1 edges.
- A graph with few edges compared with the most possible (|V|² for directed graphs) is **sparse**; one with many is **dense**. Most real graphs (roads, friendships, the web) are very sparse.

Two counting facts: in an undirected graph, the degrees add up to 2|E| (Euler's **handshake lemma**, from the warm-up); in a directed graph, in-degrees and out-degrees each add up to |E|.

## Three ways to store a graph

**An edge list**: a list of pairs (or triples, with weights). Simple, compact, and the usual way graphs arrive from files. But "who are A's neighbours?" means scanning every edge.

**An adjacency matrix**: number the vertices 0 to n − 1, and keep an n × n table where `m[i][j]` is 1 (or the weight) if there's an edge from i to j. "Is there an edge from i to j?" is O(1), but the table takes n² space even when there are few edges, and listing neighbours scans a whole row.

**An adjacency list**: for each vertex, the collection of its neighbours. In Python, a **dict of sets** (or dict of lists, or for weighted graphs a dict of dicts from neighbour to weight). Space is O(V + E), and listing a vertex's neighbours costs only as much as there are neighbours. This is the default choice.

```text
  edges: A-B, A-C, B-C, C-D

  edge list                 adjacency matrix            adjacency list
  [(A,B), (A,C),               A  B  C  D              A: {B, C}
   (B,C), (C,D)]            A  0  1  1  0              B: {A, C}
                            B  1  0  1  0              C: {A, B, D}
                            C  1  1  0  1              D: {C}
                            D  0  0  1  0
```

For a **weighted** graph, the adjacency list becomes a dict of dicts, `adj[a][b] = weight`, and the matrix holds weights (with `None` or infinity for "no edge", since 0 might be a real weight).

## Which one to use

| operation | edge list | adjacency matrix | adjacency list (dict of sets) |
|---|---|---|---|
| space | O(E) | O(V²) | O(V + E) |
| is there an edge u → v? | O(E) | O(1) | O(1) average |
| list u's neighbours | O(E) | O(V) | O(degree of u) |
| add an edge | O(1) | O(1) | O(1) average |
| visit every edge | O(E) | O(V²) | O(V + E) |

Graph algorithms mostly visit each vertex and look at its neighbours, so they cost O(V + E) with adjacency lists but O(V²) with a matrix. For a sparse graph of a million vertices and a few million edges, that's the difference between seconds and days (and a matrix of 10¹² cells wouldn't fit in memory anyway). Use a matrix only for small or dense graphs, or when constant-time edge checks matter most. Keep an edge list when the edges themselves are what you process (sorting them by weight, as next week's minimum spanning trees do).

## A graph class

A class keeps the representation and its rules together: whether edges have a direction, that adding an edge adds its vertices, that removing a vertex removes its edges.

(A self-loop, an edge from a vertex to itself, would be counted once in the undirected total and then halved; graphs in this course have none unless a problem says so.)

## Graphs you never store

Many graphs are **implicit**: you can compute a vertex's neighbours when you need them, so there's nothing to store.

- A **grid** (a maze, a map of tiles, an image): each cell's neighbours are the cells above, below, left and right that are inside the grid and not walls.
- **States of a puzzle**: each arrangement of a sliding puzzle is a vertex; one move leads to a neighbour.
- Week 49's **word ladder**: words are vertices, one-letter changes are edges.

All that's needed is a function `neighbours(v)`. Every algorithm this week works on such a function just as well as on a dict.

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a small graph from your own life: five people and who knows whom, or five places and the roads between them. Write it as an edge list, an adjacency list and an adjacency matrix. Tomorrow: breadth-first search, the way to explore a graph level by level and find shortest paths.